In [0]:
import pyspark.sql.functions as f

In [0]:
%run ../utility/read_write_util

In [0]:
df = spark.read.table('nyc_cleansed.taxi_zone_lookup')

In [0]:
rename_cols = {
    'locationid':'location_id'
}

for key,val in rename_cols.items():
    df = df.withColumnRenamed(key,val)

select_cols = [
    'location_id',
    'borough',
    'zone',
    'service_zone'
]

df = df.select(*select_cols)

In [0]:
source_df = df

Implement SCD-1

In [0]:
from delta.tables import * 

try:
    target_df = DeltaTable.forName(spark,'nyc_enterprise.taxi_zone_lookup')
    print('Target Table Exists & Successfully loaded into dataframe ')

except Exception as e :
    print(e)
    print('Error While Reading , Creating New Table ')
    target_df = spark.createDataFrame([],schema=source_df.schema)
    target_df = target_df.withColumn('created_by',f.lit('manual'))\
                         .withColumn('created_datetime',f.current_timestamp())\
                         .withColumn('modified_by',f.lit(''))\
                         .withColumn('modified_datetime',f.lit(''))

    spark.sql('CREATE SCHEMA IF NOT EXISTS nyc_enterprise')
    write_data(
        df = target_df,
        target_file_path = f"nyc_enterprise.taxi_zone_lookup",
        target_file_format='delta',
        mode_type='overwrite'
            )
    
source_df = source_df.withColumn('created_by',f.lit('data-pipeline'))\
                     .withColumn('created_datetime',f.current_timestamp())\
                     .withColumn('modified_by',f.lit(''))\
                     .withColumn('modified_datetime',f.lit(''))


target_df.alias('target').merge(
    source_df.alias('source'),
    condition =
    """
    source.location_id = target.location_id
    """
).whenMatchedUpdate(
    condition=
    """
    target.borough <> source.borough OR
    target.zone <> source.zone OR 
    target.service_zone <> source.service_zone 
    """,
    set = {
        "borough":"source.borough",
        "zone":"source.zone",
        "service_zone":"source.service_zone",
        "modified_by":"'data-pipeline'",
        "modified_datetime":"current_timestamp()"
    }
).whenNotMatchedInsert(
    values = {
        "location_id":"source.location_id",
        "borough":"source.borough",
        "zone":"source.zone",
        "service_zone":"source.service_zone",
        "created_by":"source.created_by",
        "created_datetime":"source.created_datetime",
        "modified_by":"source.modified_by",
        "modified_datetime":"source.modified_datetime"
    }

).execute()

print('Updated the Taxi Zone Look Up Table')